<a href="https://colab.research.google.com/github/keyvanmahmoudi/Python_Course/blob/main/sessions/Session13_exceptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۳: مدیریت خطا و اشکال‌زدایی

برنامه‌های واقعی با ورودی‌های نادرست، فایل‌های ناموجود و شرایط پیش‌بینی‌نشده روبه‌رو می‌شوند.
در این جلسه یاد می‌گیریم به‌جای از کار افتادن برنامه، خطاها را **مدیریت** کنیم، خودمان در جای مناسب
خطا ایجاد کنیم و با روش‌های اشکال‌زدایی، ایرادهای کد را پیدا کنیم.

### اهداف این جلسه

- انواع خطا و خواندن traceback
- ساختار `try`، `except`، `else` و `finally`
- ایجاد خطا با `raise` و ساخت استثنای سفارشی
- اعتبارسنجی ورودی کاربر
- روش‌های اشکال‌زدایی: `print`، `assert` و دیباگر

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. خطاهای نحوی و استثناها

دو نوع خطا داریم:

- **خطای نحوی (SyntaxError):** کد از قواعد پایتون پیروی نمی‌کند و اصلاً اجرا نمی‌شود. باید کد را اصلاح کرد.
- **استثنا (Exception):** کد از نظر نحوی درست است ولی هنگام اجرا مشکلی پیش می‌آید. این‌ها را می‌توان **مدیریت** کرد.

استثناهای رایج:

| استثنا | مثال |
|---|---|
| `ValueError` | `int("abc")` |
| `TypeError` | `"2" + 2` |
| `ZeroDivisionError` | `10 / 0` |
| `IndexError` | `[1, 2][5]` |
| `KeyError` | `{"a": 1}["b"]` |
| `FileNotFoundError` | `open("missing.txt")` |
| `NameError` | استفاده از متغیر تعریف‌نشده |
| `AttributeError` | `"abc".push()` |

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**traceback** را از پایین به بالا بخوانید: آخرین خط نوع و پیام خطا را می‌گوید و خطوط بالاتر مسیر فراخوانی‌ها تا محل خطا را:

</div>

In [1]:
def average(numbers):
    return sum(numbers) / len(numbers)

def report(scores):
    print("average:", average(scores))

report([])

ZeroDivisionError: division by zero

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. try و except

کدی را که ممکن است خطا بدهد در بلوک `try` می‌گذاریم. اگر خطایی رخ دهد، اجرا به بلوک `except` منتقل می‌شود
و برنامه **ادامه پیدا می‌کند**.

- همیشه **نوع مشخص** خطا را بگیرید (`except ValueError`)، نه `except:` خالی که همه‌چیز، حتی خطاهای برنامه‌نویسی
  را پنهان می‌کند.
- با `as e` می‌توان به پیام خطا دسترسی داشت.
- می‌توان چند `except` برای خطاهای مختلف داشت.

</div>

In [2]:
try:
    n = int("abc")
    print("this line is skipped")
except ValueError as e:
    print("Invalid number:", e)

print("program continues")

Invalid number: invalid literal for int() with base 10: 'abc'
program continues


In [3]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("cannot divide by zero")
        return None
    except TypeError:
        print("both values must be numbers")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "2"))

5.0
cannot divide by zero
None
both values must be numbers
None


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. else و finally

- `else`: فقط وقتی اجرا می‌شود که در `try` **هیچ خطایی** رخ نداده باشد.
- `finally`: **همیشه** اجرا می‌شود؛ چه خطا رخ بدهد چه ندهد. مناسب کارهای پایانی مثل بستن اتصال‌ها.

</div>

In [4]:
def read_number(text):
    try:
        value = float(text)
    except ValueError:
        print(f"'{text}' is not a number")
    else:
        print(f"read {value}")
        return value
    finally:
        print("-- attempt finished --")

read_number("3.5")
read_number("three")

read 3.5
-- attempt finished --
'three' is not a number
-- attempt finished --


In [5]:
try:
    with open("missing_file.txt", encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError as e:
    print("Error:", e)

Error: [Errno 2] No such file or directory: 'missing_file.txt'


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. ایجاد خطا با raise

گاهی خودمان باید خطا ایجاد کنیم؛ مثلاً وقتی تابع ورودی نامعتبر دریافت می‌کند. با `raise` یک استثنا می‌سازیم.
این کار بهتر از برگرداندن مقدارهای عجیب (مثل `-1`) است، چون خطا پنهان نمی‌ماند.

برای خطاهای خاص برنامهٔ خودمان می‌توانیم کلاس استثنای سفارشی بسازیم (کلاس‌ها را در جلسهٔ بعد کامل می‌بینیم).

</div>

In [6]:
def set_score(score):
    if not isinstance(score, (int, float)):
        raise TypeError("score must be a number")
    if not 0 <= score <= 20:
        raise ValueError(f"score must be between 0 and 20, got {score}")
    return score

for value in [18, 25, "A"]:
    try:
        print("accepted:", set_score(value))
    except (ValueError, TypeError) as e:
        print(type(e).__name__, "->", e)

accepted: 18
ValueError -> score must be between 0 and 20, got 25
TypeError -> score must be a number


In [7]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the balance."""


def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"balance {balance:,} < requested {amount:,}")
    return balance - amount

try:
    withdraw(500_000, 800_000)
except InsufficientFundsError as e:
    print("Transaction failed:", e)

Transaction failed: balance 500,000 < requested 800,000


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. اعتبارسنجی ورودی کاربر

الگوی رایج: در یک حلقهٔ `while True` ورودی بگیریم و تا وقتی ورودی معتبر نیست، دوباره بپرسیم.

</div>

In [8]:
def ask_score(prompt="Score (0-20): "):
    while True:
        text = input(prompt)
        try:
            score = float(text)
        except ValueError:
            print("  Please enter a number.")
            continue
        if 0 <= score <= 20:
            return score
        print("  Score must be between 0 and 20.")

score = ask_score()
print("Accepted score:", score)

Score (0-20): twenty
  Please enter a number.
Score (0-20): -3
  Score must be between 0 and 20.
Score (0-20): 18.5
Accepted score: 18.5


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. اشکال‌زدایی (Debugging)

وقتی برنامه خطا نمی‌دهد ولی نتیجه اشتباه است، باید **اشکال‌زدایی** کنیم. چند روش:

1. **خواندن دقیق پیام خطا** و traceback.
2. **چاپ مقدارها:** `print(f"{x=}")` نام و مقدار متغیر را با هم چاپ می‌کند.
3. **`assert`:** فرضی را که باید درست باشد بررسی می‌کند؛ اگر نادرست بود خطای `AssertionError` می‌دهد.
4. **دیباگر:** در VS Code با گذاشتن breakpoint (کلیک کنار شمارهٔ خط) و اجرای Debug، می‌توان برنامه را خط‌به‌خط
   اجرا کرد و مقدار متغیرها را دید. در Jupyter هم دستور جادویی `%debug` بعد از خطا کار می‌کند.
5. **آزمایش با ورودی‌های کوچک** و حالت‌های مرزی (لیست خالی، صفر، عدد منفی).

</div>

In [9]:
def average(numbers):
    assert len(numbers) > 0, "numbers must not be empty"
    total = sum(numbers)
    print(f"{total=} {len(numbers)=}")
    return total / len(numbers)

print(average([12, 15, 18]))
average([])

total=45 len(numbers)=3
15.0


AssertionError: numbers must not be empty

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

یک مثال واقعی از باگ منطقی: این تابع باید میانگین اعداد مثبت را برگرداند ولی نتیجه اشتباه است. با چاپ مقدارها علت را پیدا می‌کنیم:

</div>

In [10]:
def positive_average(numbers):
    total = 0
    for n in numbers:
        if n > 0:
            total += n
    print(f"{total=} {len(numbers)=}")   # debug line
    return total / len(numbers)          # bug: divides by all numbers

print(positive_average([10, -5, 20, -1]))

total=30 len(numbers)=4
7.5


In [11]:
def positive_average(numbers):
    positives = [n for n in numbers if n > 0]
    if not positives:
        raise ValueError("no positive numbers")
    return sum(positives) / len(positives)

print(positive_average([10, -5, 20, -1]))

15.0


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## جمع‌بندی

- خطای نحوی را باید اصلاح کرد؛ استثناها را می‌توان با `try`/`except` مدیریت کرد.
- همیشه نوع مشخص خطا را بگیرید و از `except:` خالی پرهیز کنید.
- `else` اگر خطایی نبود و `finally` همیشه اجرا می‌شود.
- با `raise` برای ورودی نامعتبر خطا ایجاد کنید.
- برای اشکال‌زدایی: پیام خطا، `print(f"{x=}")`، `assert` و دیباگر.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. تابع `to_int(text)` بنویسید که رشته را به عدد صحیح تبدیل کند و اگر ممکن نبود `None` برگرداند. آن را روی `["12", "x", "7", "3.5"]` امتحان کنید.
۲. از لیست `["10", "20", "abc", "30", ""]` فقط مقدارهای قابل تبدیل به عدد را جمع کنید و تعداد مقدارهای نامعتبر را گزارش دهید.
۳. تابع `get_item(lst, index)` بنویسید که اگر اندیس خارج از محدوده بود، به‌جای خطا پیام مناسب چاپ کند و `None` برگرداند.
۴. تابع `create_user(username, age)` بنویسید که اگر نام کاربری کمتر از ۳ کاراکتر بود یا سن بین ۱ و ۱۲۰ نبود، `ValueError` با پیام مناسب ایجاد کند؛ وگرنه دیکشنری کاربر را برگرداند.
۵. تابع `read_json(path)` بنویسید که فایل JSON را بخواند و اگر فایل وجود نداشت یا محتوایش JSON معتبر نبود (`json.JSONDecodeError`)، پیام مناسب چاپ کند و دیکشنری خالی برگرداند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** تابع `to_int(text)` بنویسید که رشته را به عدد صحیح تبدیل کند و اگر ممکن نبود `None` برگرداند. آن را روی `["12", "x", "7", "3.5"]` امتحان کنید.

</div>

In [12]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** از لیست `["10", "20", "abc", "30", ""]` فقط مقدارهای قابل تبدیل به عدد را جمع کنید و تعداد مقدارهای نامعتبر را گزارش دهید.

</div>

In [13]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** تابع `get_item(lst, index)` بنویسید که اگر اندیس خارج از محدوده بود، به‌جای خطا پیام مناسب چاپ کند و `None` برگرداند.

</div>

In [14]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** تابع `create_user(username, age)` بنویسید که اگر نام کاربری کمتر از ۳ کاراکتر بود یا سن بین ۱ و ۱۲۰ نبود، `ValueError` با پیام مناسب ایجاد کند؛ وگرنه دیکشنری کاربر را برگرداند.

</div>

In [15]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۵.** تابع `read_json(path)` بنویسید که فایل JSON را بخواند و اگر فایل وجود نداشت یا محتوایش JSON معتبر نبود (`json.JSONDecodeError`)، پیام مناسب چاپ کند و دیکشنری خالی برگرداند.

</div>

In [16]:
# پاسخ خود را اینجا بنویسید
